# 02 PyTorch 数据读取：Dataset 与 DataLoader

这一节只讲一件事：**数据怎么喂给模型**。

上一节你学的是 Tensor：一个样本、一批数据、模型参数都可以用 Tensor 表示。

但真正训练模型时，我们不会一次把所有数据都扔进去，而是通常这样做：

1. 把所有样本组织成一个 `Dataset`。
2. 用 `DataLoader` 每次取出一小批样本，也就是一个 batch。
3. 后面模型就一批一批地训练。

本节暂时不讲 MLP、不讲损失函数、不讲优化器。先把“数据如何变成 batch”学清楚。

## 0. 环境准备

推荐环境：`D:\\PythonWorkSpace\\anaconda\\envs\\pytorch\\python.exe`。

本节只用 CPU，不涉及 GPU。

In [ ]:
import torch
from torch.utils.data import Dataset, TensorDataset, DataLoader, random_split

torch.manual_seed(42)

print("torch version:", torch.__version__)

## 1. 先说人话：为什么需要 Dataset 和 DataLoader

假设你有 1000 条训练数据。

直接把 1000 条一次性喂给模型，有几个问题：

- 数据量大时，内存可能不够。
- 每次更新参数都看完整数据，训练会慢。
- 模型每次看到数据顺序都一样，训练效果可能不好。

所以深度学习通常使用 mini-batch 训练：每次只取一小批，比如 32 条。

这里就需要两个工具：

| 工具 | 人话解释 | 负责什么 |
|---|---|---|
| `Dataset` | 数据仓库 | 告诉 PyTorch 有多少样本、每个样本怎么取 |
| `DataLoader` | 数据搬运工 | 从 Dataset 里按 batch 取数据，可打乱、可并行加载 |

一句话：`Dataset` 管“数据长什么样”，`DataLoader` 管“怎么一批一批拿出来”。

## 2. 准备一份最小数据

我们先不用真实图片或文本，先造一个最简单的表格数据。

任务背景：根据一个学生的两个特征，预测是否通过考试。

- `x`：输入特征，形状是 `[样本数, 特征数]`。
- `y`：标签，形状是 `[样本数]`。

这里的标签先用 0/1 表示：

- `0`：未通过
- `1`：通过

建议你从这一格开始按顺序运行。后面个别关键单元会自带最小数据，方便你单独试。


In [ ]:
# 6 个样本，每个样本 2 个特征。
# 你可以把两个特征理解成：学习时长、练习题正确率。
x = torch.tensor([
    [1.0, 0.20],
    [2.0, 0.30],
    [3.0, 0.50],
    [4.0, 0.60],
    [5.0, 0.80],
    [6.0, 0.90]
])

y = torch.tensor([0, 0, 0, 1, 1, 1])

print("x =\n", x)
print("y =", y)
print("x.shape =", x.shape)
print("y.shape =", y.shape)

## 3. `TensorDataset`：最简单的 Dataset

### 3.1 作用

`TensorDataset` 可以把一个或多个 Tensor 包装成 Dataset。

最常见写法：

```python
dataset = TensorDataset(x, y)
```

人话：第 0 个样本就是 `(x[0], y[0])`，第 1 个样本就是 `(x[1], y[1])`，以此类推。

### 3.2 参数

| 参数 | 说明 |
|---|---|
| `*tensors` | 一个或多个 Tensor，比如 `x, y` |

要求：传进去的 Tensor 第 0 维长度必须相同。

例如：

- `x.shape = [6, 2]`
- `y.shape = [6]`

它们第 0 维都是 6，表示都有 6 个样本，所以可以配对。

In [ ]:
import torch
from torch.utils.data import TensorDataset

# 为了避免单独运行这一格时报 NameError，这里把本节用到的最小数据也放进来。
x = torch.tensor([
    [1.0, 0.20],
    [2.0, 0.30],
    [3.0, 0.50],
    [4.0, 0.60],
    [5.0, 0.80],
    [6.0, 0.90]
])

y = torch.tensor([0, 0, 0, 1, 1, 1])

dataset = TensorDataset(x, y)

print("数据集中有多少个样本:", len(dataset))
print("第 0 个样本:", dataset[0])
print("第 3 个样本:", dataset[3])


这里的 `dataset[0]` 返回的是一个元组：

```python
(第 0 个样本的特征, 第 0 个样本的标签)
```

也就是：

```python
(x[0], y[0])
```

## 4. `DataLoader`：按 batch 取数据

### 4.1 作用

`DataLoader` 用来从 Dataset 中批量取数据。

最常见写法：

```python
loader = DataLoader(dataset, batch_size=2, shuffle=True)
```

### 4.2 常用参数

| 参数 | 人话解释 | 常见取值 |
|---|---|---|
| `dataset` | 从哪个数据集取数据 | 前面创建的 `dataset` |
| `batch_size` | 每次取几个样本 | 16、32、64、128 |
| `shuffle` | 每轮是否打乱样本顺序 | 训练集常用 `True` |
| `drop_last` | 最后不够一个 batch 时是否丢掉 | 数据很多时可用 `True` |
| `num_workers` | 用几个子进程加载数据 | Windows 初学先用 `0` |

初学阶段先记住三个：`dataset`、`batch_size`、`shuffle`。

In [ ]:
loader = DataLoader(dataset, batch_size=2, shuffle=False)

for batch_index, (batch_x, batch_y) in enumerate(loader):
    print("batch", batch_index)
    print("batch_x =\n", batch_x)
    print("batch_y =", batch_y)
    print("batch_x.shape =", batch_x.shape)
    print("batch_y.shape =", batch_y.shape)
    print("-" * 30)

观察上面的输出：

- 原来 `x.shape = [6, 2]`。
- 设置 `batch_size=2` 后，每次取出来的 `batch_x.shape = [2, 2]`。
- 第一维 `2` 是 batch 里的样本数。
- 第二维 `2` 是每个样本的特征数。

## 5. `shuffle=True`：为什么训练集要打乱

如果数据是按类别排好的，比如前 3 个是 0 类，后 3 个是 1 类，不打乱时模型会先连续看到一堆 0，再连续看到一堆 1。

训练时通常希望每个 batch 尽量随机一些，所以训练集常用：

```python
shuffle=True
```

验证集、测试集通常不需要打乱，因为我们只是评估，不更新模型参数。

In [ ]:
loader_no_shuffle = DataLoader(dataset, batch_size=2, shuffle=False)
loader_shuffle = DataLoader(dataset, batch_size=2, shuffle=True)

print("不打乱:")
for batch_x, batch_y in loader_no_shuffle:
    print(batch_y.tolist())

print("\n打乱:")
for batch_x, batch_y in loader_shuffle:
    print(batch_y.tolist())

## 6. `drop_last`：最后一个 batch 不够怎么办

如果有 5 个样本，`batch_size=2`：

- 第 1 批：2 个
- 第 2 批：2 个
- 第 3 批：1 个

`drop_last=False`：保留最后 1 个样本。

`drop_last=True`：丢掉最后不完整的 batch。

初学阶段一般先用默认值 `False`。

In [ ]:
small_x = torch.arange(10).reshape(5, 2).float()
small_y = torch.tensor([0, 1, 0, 1, 0])
small_dataset = TensorDataset(small_x, small_y)

print("drop_last=False:")
loader_keep = DataLoader(small_dataset, batch_size=2, drop_last=False)
for batch_x, batch_y in loader_keep:
    print(batch_x.shape, batch_y.tolist())

print("\ndrop_last=True:")
loader_drop = DataLoader(small_dataset, batch_size=2, drop_last=True)
for batch_x, batch_y in loader_drop:
    print(batch_x.shape, batch_y.tolist())

## 7. 自定义 Dataset：真实项目更常见

`TensorDataset` 适合数据已经全部变成 Tensor 的情况。

但真实项目里，数据可能是：

- 图片文件路径
- 文本字符串
- CSV 中的一行
- 音频文件

这时就要自己写 Dataset 类。

自定义 Dataset 必须实现两个方法：

| 方法 | 必须做什么 |
|---|---|
| `__len__(self)` | 返回数据集中有多少个样本 |
| `__getitem__(self, index)` | 根据索引返回一个样本 |

人话：`__len__` 回答“仓库里有多少件货”，`__getitem__` 回答“第 index 件货是什么”。

In [ ]:
class ExamDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        feature = self.features[index]
        label = self.labels[index]
        return feature, label


custom_dataset = ExamDataset(x, y)

print("样本数量:", len(custom_dataset))
print("第 0 个样本:", custom_dataset[0])

现在这个 `custom_dataset` 和前面的 `TensorDataset` 用起来几乎一样，也可以交给 `DataLoader`。

In [ ]:
custom_loader = DataLoader(custom_dataset, batch_size=3, shuffle=False)

for batch_x, batch_y in custom_loader:
    print("batch_x =\n", batch_x)
    print("batch_y =", batch_y)
    print("-" * 30)

## 8. 训练集、验证集、测试集怎么分

机器学习里你应该已经见过数据集划分。深度学习里也一样。

| 数据集 | 作用 |
|---|---|
| 训练集 train | 用来更新模型参数 |
| 验证集 validation | 训练过程中用来观察模型效果、调超参数 |
| 测试集 test | 最后只用一次，用来估计最终泛化能力 |

PyTorch 可以用 `random_split` 随机划分 Dataset。

In [ ]:
train_size = 4
val_size = 1
test_size = 1

generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=generator
)

print("train:", len(train_dataset))
print("val:", len(val_dataset))
print("test:", len(test_dataset))
print("训练集第 0 个样本:", train_dataset[0])

`random_split` 常用参数：

| 参数 | 说明 |
|---|---|
| `dataset` | 要划分的数据集 |
| `lengths` | 每一份的长度，比如 `[4, 1, 1]` |
| `generator` | 随机数生成器，用来固定划分结果 |

为什么要固定 `generator`？

因为学习和实验时，你希望每次运行划分结果一样，方便比较。

## 9. 给不同数据集创建不同 DataLoader

常见写法：

- 训练集：`shuffle=True`
- 验证集：`shuffle=False`
- 测试集：`shuffle=False`

原因：训练时需要随机性，评估时只需要稳定地算指标。

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False)

print("训练集 batch:")
for batch_x, batch_y in train_loader:
    print(batch_x, batch_y)

print("\n验证集 batch:")
for batch_x, batch_y in val_loader:
    print(batch_x, batch_y)

print("\n测试集 batch:")
for batch_x, batch_y in test_loader:
    print(batch_x, batch_y)

## 10. 一个容易混的点：样本 shape 和 batch shape

单个样本：

```python
feature.shape = [2]
label.shape = []
```

一个 batch：

```python
batch_x.shape = [batch_size, 2]
batch_y.shape = [batch_size]
```

`DataLoader` 会自动把多个样本堆叠成 batch。

这就是为什么模型里第一维通常是 batch 维。

In [ ]:
one_feature, one_label = dataset[0]
print("单个样本 feature.shape:", one_feature.shape)
print("单个样本 label.shape:", one_label.shape)

batch_x, batch_y = next(iter(DataLoader(dataset, batch_size=4)))
print("batch_x.shape:", batch_x.shape)
print("batch_y.shape:", batch_y.shape)

## 11. 本节必须掌握的总结

### 11.1 Dataset 是什么

`Dataset` 是数据集对象，它至少要能回答两个问题：

1. 这个数据集有多少个样本？对应 `len(dataset)`。
2. 第 `index` 个样本是什么？对应 `dataset[index]`。

### 11.2 DataLoader 是什么

`DataLoader` 是批量读取器，它从 Dataset 里取样本，并自动组成 batch。

### 11.3 什么时候用 TensorDataset

如果你的特征和标签已经是 Tensor，可以直接用：

```python
dataset = TensorDataset(x, y)
```

### 11.4 什么时候自定义 Dataset

如果你的数据还在图片文件、文本文件、CSV、数据库里，通常要自定义 Dataset。

### 11.5 DataLoader 最常用参数

| 参数 | 记忆方式 |
|---|---|
| `dataset` | 从哪拿数据 |
| `batch_size` | 每次拿几个 |
| `shuffle` | 是否打乱顺序 |
| `drop_last` | 最后一批不够时要不要丢 |
| `num_workers` | 找几个工人一起读数据，Windows 初学先用 0 |

## 12. 自检题

学完本节，先别急着训练模型。你应该能回答：

1. `Dataset` 和 `DataLoader` 分别负责什么？
2. `TensorDataset(x, y)` 对 `x` 和 `y` 的形状有什么要求？
3. `batch_size=32` 是什么意思？
4. 为什么训练集通常 `shuffle=True`？
5. 验证集和测试集为什么通常不需要 shuffle？
6. `drop_last=True` 会发生什么？
7. 自定义 Dataset 为什么必须写 `__len__` 和 `__getitem__`？
8. 单个样本 shape 和 batch shape 有什么区别？

## 13. 小练习

请你自己改代码试试：

1. 把 `batch_size=2` 改成 `3`，观察 batch 数量和 shape 怎么变。
2. 把 `shuffle=False` 改成 `True`，连续运行两次，看 batch 顺序是否变化。
3. 把 `small_dataset` 的样本数从 5 改成 7，再观察 `drop_last=True` 和 `False` 的区别。
4. 给 `ExamDataset` 多加一个字段，比如学生姓名，看看 `__getitem__` 可以返回几个东西。

下一节再讲 `nn.Module`：如何定义一个真正的神经网络模型。